In [1]:
import torch
import re
from collections import defaultdict
import math

In [2]:
text = open('train.txt', encoding='utf-8').read()
print(f"Loaded: {len(text):,} chars, ~{len(text.split()):,} words")


Loaded: 3,808,272 chars, ~658,747 words


In [3]:
# text is already loaded from the download cell above

unique_chars = sorted(list(set(text)))
vocab_size = len(unique_chars)


print(''.join(unique_chars))
print(f"\nvocab_Size:{vocab_size}")

# unique_chars.extend(['4','6','7','9','0'])
# Adding remaining numbers

def word_tokenize(text):
    """Each word is a token, but also each special character, spaces arent ocnsidered."""
    # Split on spaces but keep punctuation as separate tokens
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens
word_tokens = word_tokenize(text)


print(f"\nFirst 50 WORD tokens:\n {word_tokens[:50]}")




 !&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz£º½àâæçèéêëîïñôöûüœ—‘’“”・

vocab_Size:104

First 50 WORD tokens:
 ['EBOOK', 'A', 'STUDY', 'IN', 'SCARLET', '*', '*', '*', 'A', 'STUDY', 'IN', 'SCARLET', 'By', 'A', '.', 'Conan', 'Doyle', 'CONTENTS', 'A', 'STUDY', 'IN', 'SCARLET', '.', 'PART', 'I', '.', 'CHAPTER', 'I', '.', 'MR', '.', 'SHERLOCK', 'HOLMES', '.', 'CHAPTER', 'II', '.', 'THE', 'SCIENCE', 'OF', 'DEDUCTION', '.', 'CHAPTER', 'III', '.', 'THE', 'LAURISTON', 'GARDENS', 'MYSTERY', 'CHAPTER']


In [4]:
def split_words_with_sep(text):
    for m in re.finditer(r'(\S+)(\s*)', text):
        word, sep = m.groups()
        if not word:
            continue
        marker = '<para>' if sep.count('\n') >= 2 else '<end>'
        yield word, marker

# Load pretrained BPE vocab + merges (from bpe_train.py) instead of training in-notebook
with open('tokens.txt', encoding='utf-8') as f:
    bpe_tokens = [line.rstrip('\n') for line in f]

with open('merges.txt', encoding='utf-8') as f:
    merges = [tuple(line.rstrip('\n').split('\t')) for line in f]

print(f"BPE vocab size: {len(bpe_tokens)}")
print(f"First 20 tokens: {bpe_tokens[:20]}")

stoi = {token: i for i, token in enumerate(bpe_tokens)}
itos = {i: token for i, token in enumerate(bpe_tokens)}

bpe_vocab_size=len(bpe_tokens)
#string ot integer and integer to string

def encode(text):
    token_ids = []
    for word, marker in split_words_with_sep(text):
        symbols = list(word) + [marker]
        for pair in merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                    symbols = symbols[:i] + [''.join(pair)] + symbols[i+2:]
                else:
                    i += 1
        for token in symbols:
            if token in stoi:
                token_ids.append(stoi[token])
    return token_ids

encoded = encode(text)

def decode(token_ids):
    tokens = [itos[i] for i in token_ids]
    text = ''.join(tokens)
    text = text.replace('<para>', '\n\n').replace('<end>', ' ')
    return text.strip()

n = int(0.9 * len(encoded))
train = encoded[:n]
test = encoded[n:]


train =  torch.tensor(train, dtype=torch.long)
test = torch.tensor(test, dtype=torch.long)

print(f"Train tokens: {len(train)}, Test tokens: {len(test)}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


BPE vocab size: 1103
First 20 tokens: ['!', '!<end>', '!”<end>', '!”<para>', '&', "'", '(', ')', '*', ',', ',<end>', ',<para>', ',’<end>', ',”<end>', '-', '.', '.<end>', '.<para>', '.’<para>', '.”<para>']
Train tokens: 1109558, Test tokens: 123285
Device: Tesla T4


In [5]:
torch.manual_seed(10)
block_size = 256  # context length
batch_size = 16  # reduced from 32 — block_size=128 doubles memory per sample

def get_batch(data):
    #random start positions
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # x is the input context, y is the target (shifted by 1)
    x = torch.stack([data[i : i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])

    return x, y

xb, yb = get_batch(train)
print(xb.shape)
print(yb.shape,"\n\n")

print("INPUTS",xb)
print("OUTPUTS:",yb)



torch.Size([16, 256])
torch.Size([16, 256]) 


INPUTS tensor([[935, 599, 550,  ..., 326, 771, 522],
        [537, 699, 457,  ..., 136, 666,  10],
        [794, 715, 672,  ..., 203, 478, 967],
        ...,
        [960,  34, 903,  ..., 366, 457, 515],
        [515, 504, 683,  ..., 254, 976, 244],
        [900,  66, 468,  ..., 167, 516, 481]])
OUTPUTS: tensor([[599, 550,  11,  ..., 771, 522, 795],
        [699, 457, 265,  ..., 666,  10, 903],
        [715, 672, 495,  ..., 478, 967, 903],
        ...,
        [ 34, 903, 784,  ..., 457, 515, 978],
        [504, 683,  48,  ..., 976, 244, 526],
        [ 66, 468, 260,  ..., 516, 481, 610]])


In [6]:
torch.manual_seed(10)
import torch.nn as nn

import torch.nn.functional as F
n_embed=256
dropout=0.2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
# Self Attention Head:
# Every token spits out two vectors, one is the key vector and the other is the query vector. The key vector is used to represent the token itself,
# the query vector is used to represent the token's relationship with prev toks.
# Query . Key(eachtoken) gives a score that represents how much attention should be paid to each token when predicting the next token.
#  score is softmaxxed to get PDF over all tokens.
# This is  used to compute a weighted sum of the value vectors of all tokens, which gives the final output for that token.

# weight matrix becomes that once our head is done.

In [8]:
#Attention HEAD.
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Scale by sqrt(head_size) — not sqrt(C) — so the scale matches the
        # dimension being dot-producted (head_size), not the full embedding dim.
        weight = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)  # (B, T, T)
        weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weight = F.softmax(weight, dim=-1)                             # (B, T, T)
        weight = self.dropout(weight)

        v = self.value(x)  # (B, T, head_size)
        return weight @ v  # (B, T, head_size)

In [9]:
# Multi-Head Self-Attention
# Run num_heads independent attention heads in parallel, each in a
# head_size = n_embed // num_heads subspace.
# Concatenate their outputs (restoring n_embed width) and project back.
# Each head can specialise on different positional / semantic relationships.

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Each head: (B, T, head_size); cat along last dim → (B, T, n_embed)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))  # (B, T, n_embed)




# Repetitive blocks of self attention and feed forward layers:

# Feed-Forward: per-token MLP, as per the "Attention Is All You Need" paper.
# Expands to 4x n_embed, applies a nonlinearity, then projects back down to
# n_embed so it can be added back into the residual stream.
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embed, n_head):
        # n_embed: embedding dimension, n_head: the number of heads
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))#communication
        x = x + self.ffwd(self.ln2(x))#computation
        return x

In [10]:
class BigramLM(nn.Module):

    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        # 4 heads, each of size n_embed // 4
        self.blocks=nn.Sequential(
            Block(n_embed, n_head=4),
            Block(n_embed, n_head=4)
        )  # 2 blocks
        self.ln_f      = nn.LayerNorm(n_embed)
        self.lm_head   = nn.Linear(n_embed, vocab_size)
        self.token_embedding_table.weight = self.lm_head.weight  # weight tying

    def forward(self, index, targets=None):
        tok_emb = self.token_embedding_table(index)                                                # (B, T, n_embed)
        pos_emb = self.position_embedding_table(torch.arange(index.size(1), device=index.device)) # (T, n_embed)
        x       = tok_emb + pos_emb # (B, T, n_embed=C)

        x       = self.blocks(x)    # (B, T, n_embed)
        x       = self.ln_f(x)      # (B, T, n_embed)
        logits  = self.lm_head(x)   # (B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss    = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens, temperature=1.0, top_k=50):
        for _ in range(max_new_tokens):
            index_cond  = index[:, -block_size:]
            logits, _   = self.forward(index_cond)
            logits      = logits[:, -1, :] / temperature
            # top-k: zero out all logits outside the k highest, then softmax
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs       = F.softmax(logits, dim=-1)
            next_token  = torch.multinomial(probs, num_samples=1)
            index       = torch.cat([index, next_token], dim=1)
        return index

model = BigramLM(bpe_vocab_size, n_embed).to(DEVICE)
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([4096, 1103])
tensor(7.1452, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
max_lr    = 1e-3
min_lr    = 1e-4
warmup    = 200
max_steps = 25000

def get_lr(step):
    # linear warmup then cosine decay
    if step < warmup:
        return max_lr * step / warmup
    t = (step - warmup) / (max_steps - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * t))

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=1e-1)
best_loss = 3.9496
eval_iters = 50  # number of batches to average over when estimating loss

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split, data in [('train', train), ('val', test)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(data)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

for i in range(25000):
    xb, yb = get_batch(train)
    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    lr = get_lr(i)
    for g in optimizer.param_groups:
        g['lr'] = lr
    optimizer.step()

    if i % 500 == 0:
        losses = estimate_loss()
        print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        if losses['val'] < best_loss:
            best_loss = losses['val']
            torch.save(model.state_dict(), 'bigram_weights.pth')

losses = estimate_loss()
print(f"Training done. Best loss: {best_loss:.4f} | final train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


step 0: train loss 7.1588, val loss 7.1577
step 500: train loss 6.3356, val loss 6.3381
step 1000: train loss 6.3240, val loss 6.3442
step 1500: train loss 6.3258, val loss 6.3312
step 2000: train loss 5.7074, val loss 5.7754
step 2500: train loss 5.3349, val loss 5.4847
step 3000: train loss 5.0853, val loss 5.3074
step 3500: train loss 4.9053, val loss 5.1607
step 4000: train loss 4.8046, val loss 5.0754
step 4500: train loss 4.7089, val loss 5.0049
step 5000: train loss 4.6109, val loss 4.9234
step 5500: train loss 4.5519, val loss 4.8861
step 6000: train loss 4.4885, val loss 4.8267
step 6500: train loss 4.4498, val loss 4.7896
step 7000: train loss 4.3881, val loss 4.7594
step 7500: train loss 4.3392, val loss 4.7286
step 8000: train loss 4.3003, val loss 4.6844
step 8500: train loss 4.2551, val loss 4.6751
step 9000: train loss 4.2082, val loss 4.6448
step 9500: train loss 4.1729, val loss 4.5962
step 10000: train loss 4.1127, val loss 4.5734
step 10500: train loss 4.0718, val lo

In [12]:
# To capture more info about previous tokens, we can use multiple ways.
#  We can aggregate the mean of all previous tokens instead of just using the previous token for predicting the next token,

# xagg=torch.zeros((B,T,C))
# for b in range(B):
#     for t in range(T):
#         xprev=x[b,:t+1,:]
#         xagg[b,t]=torch.mean(xprev,dim=0)

# xagg[b,t] stores all previous info till t (excluding t). This is a simple way to give the model more context, but it can be improved.

# print(x[0],"\n\n",xagg[0])



#This can be done using matrix multiplicaiton or softmax in a much more efficient matter instead of adding.

In [13]:
# torch.manual_seed(10)
# B,T,C=4,8,32
# x=torch.randn(B,T,C)
# x.shape

# tril=torch.tril(torch.ones(T,T))
# weight=torch.zeros((T,T))
# weight=weight.masked_fill(tril==0,float('-inf'))  # future values are nerfed because we only concentrate till present token.
# weight=F.softmax(weight,dim=-1)
# out=weight@x

# out.shape

In [34]:
model.load_state_dict(torch.load('bigram_weights.pth', map_location=DEVICE))
model.eval()

context = torch.tensor([encode("My dear watson,")], dtype=torch.long, device=DEVICE)
generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))

My dear watson, you will say, for we will

have made your room, and I was consider that I have gone. Why should you be

in the morning. I shall be getting the mountain malway!”

“How do you see thousand a title

account on which are subject, of course, with my husband would be saw

intimate and you don’t suppose that you may do

you come in that you will excuse me until I have read the investigant

of your packet, and I can give it, but I

have no doubt it, but I might be the client the most uncular matter. Now, and I have a good hansom that

the matter—that I am sure that if you had gone barrect to

conceal of it. I leave your business for yourself, and I
